# Chequeo local de métricas del Exploiter

Corre el pipeline real del Exploiter -- categorías heurísticas + el
`AlquimiaConnector` real (el asistente "monotributo" de verdad) + el juez
real (`HFRouterJudge`, SI/NO por logprobs vía hf_router) + `RealismScorer`
(TF-IDF) + `enforce_ctheta`/`score_category` reales -- **sin GPU, sin LoRA,
sin AWS**. Todo corre en tu laptop.

Reusa `_eval_category`/`_gather_eval` de `reinforce.py` (código de Alex) en
vez de reimplementar el scoring, así que los números son exactamente los
mismos que produciría una corrida real, solo que con categorías fijas
(heurísticas) en vez del generador entrenado con LoRA.

**Requisito único**: `torch`/`transformers`/`peft` instalados (CPU, sin CUDA
-- solo para satisfacer los imports de `generators.py`, nunca se carga un
modelo real). Si falta `peft`: `uv pip install --python .venv/bin/python3 peft`
en el venv que estés usando.

## Armar el pipeline

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

from roastme.config import get_settings, load_experiment_config, resolve_path
from roastme.connectors.factory import build_connector
from roastme.exploiter.ctheta import enforce_ctheta
from roastme.exploiter.generators import FrozenQueryGenerator, HeuristicCategoryGenerator
from roastme.exploiter.graders import ContractGrader
from roastme.exploiter.realism import RealismScorer
from roastme.exploiter.reinforce import _eval_category
from roastme.profile.load import load_prior, load_principles, load_profile
from _hf_router_judge import HFRouterJudge, _looks_like_api_error

pd.set_option('display.max_colwidth', 90)

cfg = load_experiment_config(ROOT / 'configs' / 'reinforce_from_profiler.yaml')
settings = get_settings()
settings.connector = 'alquimia'  # explicito: nunca pasa cerca del mock de dry_run

connector = build_connector(settings)
profile = load_profile(cfg.profile_path)
contract = load_principles(cfg.principles_path)
prior = load_prior(cfg.prior_path)
realism = RealismScorer(prior, embedding_model_name=cfg.embedding_model_name)
query_gen = FrozenQueryGenerator(
    cfg.model_name_or_path,  # nunca se carga: use_model=False mas abajo
    prior=prior,
    realism_scorer=realism,
    realism_delta=cfg.realism_delta,
    max_resample=cfg.query_max_resample,
    few_shot_k=cfg.prior_few_shot_k,
)
judge = HFRouterJudge(cfg.judge_model_name_or_path, hf_token=settings.hf_token)
grader = ContractGrader(contract, kb_root=str(resolve_path(cfg.kb_root)), use_heuristic=False, judge_fn=judge)
cat_gen = HeuristicCategoryGenerator()

print('pipeline armado. juez:', cfg.judge_model_name_or_path)

## Correr el chequeo

Cambiá `N_CATEGORIAS` para probar con más o menos (cada una hace
`queries_per_category` consultas reales al asistente + al juez -- con el
default de la config son 3 consultas por categoría).

In [ ]:
N_CATEGORIAS = 6

bundle = cat_gen.sample(profile, N_CATEGORIAS)
resultados = []

# Jupyter ya corre su propio loop de asyncio -- por eso 'await' directo en la
# celda (soportado nativamente), no asyncio.run() (que falla con
# 'cannot be called from a running event loop' dentro de un kernel).
for cat in bundle.categories:
    projected, in_ctheta = enforce_ctheta(cat, profile, eta=cfg.ctheta_eta)
    record = await _eval_category(
        projected, profile, query_gen, connector, grader, cfg,
        use_model_queries=False, realism=realism,
    )
    resultados.append({'category': projected, 'in_ctheta': in_ctheta, 'record': record})

print(f'{len(resultados)} categorias evaluadas')

## Resumen por categoría

In [ ]:
resumen = pd.DataFrame([
    {
        'categoria': r['category'].to_prompt()[:70],
        'in_ctheta': r['in_ctheta'],
        'S(c)': round(r['record'].s, 4),
        'phi_hat': round(r['record'].phi_hat, 4),
        'se': round(r['record'].se, 4),
        'Db': round(r['record'].db, 4),
        'realism_ok': r['record'].realism_ok,
        'pasa_tau': r['record'].s >= cfg.score_threshold_tau,
    }
    for r in resultados
]).sort_values('S(c)', ascending=False)
resumen

## Detalle: consultas y respuestas

In [ ]:
detalle = pd.DataFrame([
    {
        'categoria': r['category'].to_prompt()[:40],
        'query': q,
        'respuesta': (resp.strip()[:150] + '...') if len(resp.strip()) > 150 else resp.strip(),
        'violacion': round(v, 3),
    }
    for r in resultados
    for q, resp, v in zip(r['record'].queries, r['record'].responses, r['record'].violations)
])
detalle

## Gráfico: S(c) por categoría

In [ ]:
resumen.set_index('categoria')['S(c)'].plot(kind='barh', figsize=(8, 4), title='S(c) por categoría')

## Resumen de coherencia

Lo que hay que mirar antes de decidir si vale la pena escalar esto a una
corrida entrenada en AWS:

In [ ]:
todas_respuestas = [resp for r in resultados for resp in r['record'].responses]
n_malas = sum(1 for resp in todas_respuestas if not resp.strip() or _looks_like_api_error(resp))
n_en_ctheta = sum(1 for r in resultados if r['in_ctheta'])
n_realismo_ok = sum(1 for r in resultados if r['record'].realism_ok)

print(f"respuestas vacias/con forma de error: {n_malas}/{len(todas_respuestas)} "
      f"({100 * n_malas / max(len(todas_respuestas), 1):.1f}%)")
print(f"categorias dentro de C_theta sin proyeccion: {n_en_ctheta}/{len(resultados)}")
print(f"categorias dentro del presupuesto de realismo (Db<={cfg.realism_delta}): "
      f"{n_realismo_ok}/{len(resultados)}")

## Notas

- Esto usa categorías **heurísticas fijas**, no el generador entrenado con
  LoRA -- es un chequeo de que el pipeline de métricas funciona y da
  señal coherente, no una corrida real de búsqueda.
- Con pocas consultas por categoría, la varianza entre corridas es
  esperable (el asistente es estocástico) -- por eso `S(c)` penaliza la
  inconsistencia en vez de usar el promedio crudo.
- Volver a correr la celda de "Correr el chequeo" repite todo con
  consultas nuevas -- útil para ver cuánto varía de una vez a la otra.